In [1]:
import sys
import os
import glob

print("--- DIAGNOSTYKA I NAPRAWA ---")

# 1. Ustawiamy ścieżki (potwierdzone Twoim screenem)
spark_home = "/usr/local/spark"
os.environ["JAVA_HOME"] = "/usr/local/java"
os.environ["SPARK_HOME"] = spark_home

# 2. Szukamy pliku py4j (serce komunikacji z Javą)
# W Spark 3.1.2 powinien to być py4j-0.10.9-src.zip
py4j_path = glob.glob(os.path.join(spark_home, 'python', 'lib', 'py4j-*-src.zip'))

if not py4j_path:
    print("BŁĄD KRYTYCZNY: Nie widzę pliku py4j w folderze:", os.path.join(spark_home, 'python', 'lib'))
    # Sprawdzamy co tam w ogóle jest
    print("Zawartość folderu lib:", os.listdir(os.path.join(spark_home, 'python', 'lib')))
else:
    print(f"Znaleziono sterownik Javy: {py4j_path[0]}")
    
    # 3. Dodajemy te pliki do Pythona "na siłę" (przed wszystkim innym)
    sys.path.insert(0, py4j_path[0])                 # Dodaj zipa
    sys.path.insert(0, os.path.join(spark_home, 'python')) # Dodaj folder python
    
    # 4. Dopiero teraz importujemy Sparka
    try:
        from pyspark.sql import SparkSession
        
        spark = SparkSession.builder \
            .appName("StaticDataParquet") \
            .master("local[*]") \
            .getOrCreate()
            
        print("\nSUKCES! Spark działa.")
        print("Wersja Sparka:", spark.version)
        print("Katalog domowy Sparka:", os.environ["SPARK_HOME"])
        
    except Exception as e:
        print("\nNadal błąd przy uruchamianiu sesji:")
        print(e)

--- DIAGNOSTYKA I NAPRAWA ---
Znaleziono sterownik Javy: /usr/local/spark/python/lib/py4j-0.10.9-src.zip


26/01/10 22:19:09 WARN util.Utils: Your hostname, node1 resolves to a loopback address: 127.0.0.1; using 10.0.2.15 instead (on interface enp0s3)
26/01/10 22:19:09 WARN util.Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/01/10 22:19:09 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).



SUKCES! Spark działa.
Wersja Sparka: 3.1.2
Katalog domowy Sparka: /usr/local/spark


In [7]:
# Uzyskujemy dostęp do systemu plików HDFS przez "wnętrza" Sparka
fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
Path = spark._jvm.org.apache.hadoop.fs.Path

# --- KONFIGURACJA ---
input_dir = "/user/vagrant/static_data"            # Skąd bierzemy CSV
output_base_dir = "/user/vagrant/static_data_parquet" # Gdzie wrzucamy Parquet

print(f"--- Skanowanie folderu HDFS: {input_dir} ---")

try:
    # Pobieramy listę wszystkich plików w folderze
    file_list = fs.listStatus(Path(input_dir))
    
    for file_status in file_list:
        # Pobieramy pełną ścieżkę (np. hdfs://.../plik.csv) i samą nazwę
        full_path = file_status.getPath().toString()
        filename = file_status.getPath().getName()
        
        # Filtrujemy: Bierzemy tylko pliki CSV, pomijamy foldery i pliki systemowe (np. _SUCCESS)
        if file_status.isDirectory() or not filename.endswith(".csv"):
            continue
            
        print(f"\nZnaleziono plik: {filename}")
        
        # Generujemy ścieżkę wyjściową (taka sama nazwa, ale bez .csv)
        # Np. dla 'akcje.csv' powstanie folder '.../akcje'
        name_no_ext = filename.replace(".csv", "")
        output_path = f"{output_base_dir}/{name_no_ext}"
        
        try:
            # 1. Wczytanie
            # Spark sam "ogarnie" że full_path to ścieżka HDFS
            df = spark.read.csv(full_path, header=True, inferSchema=True)
            
            # 2. Zapis
            print(f"   -> Zapisywanie do: {output_path} ...")
            df.write.mode("overwrite").parquet(output_path)
            print("   -> SUKCES!")
            
        except Exception as e:
            print(f"   -> BŁĄD przy przetwarzaniu {filename}: {e}")

    print("\n--- Zakończono automatyczną konwersję! ---")

except Exception as main_e:
    print(f"Błąd dostępu do folderu HDFS: {main_e}")
    print("Upewnij się, że folder istnieje i masz do niego uprawnienia.")

--- Skanowanie folderu HDFS: /user/vagrant/static_data ---

Znaleziono plik: company_earnings.csv
   -> Zapisywanie do: /user/vagrant/static_data_parquet/company_earnings ...
   -> SUKCES!

Znaleziono plik: company_info.csv
   -> Zapisywanie do: /user/vagrant/static_data_parquet/company_info ...
   -> SUKCES!

Znaleziono plik: eohd_historical_data.csv
   -> Zapisywanie do: /user/vagrant/static_data_parquet/eohd_historical_data ...
   -> SUKCES!

Znaleziono plik: fundamental_data.csv
   -> Zapisywanie do: /user/vagrant/static_data_parquet/fundamental_data ...
   -> SUKCES!

Znaleziono plik: macro_2025_data.csv
   -> Zapisywanie do: /user/vagrant/static_data_parquet/macro_2025_data ...
   -> SUKCES!

Znaleziono plik: market_context.csv
   -> Zapisywanie do: /user/vagrant/static_data_parquet/market_context ...
   -> SUKCES!

--- Zakończono automatyczną konwersję! ---


In [8]:
# --- CZĘŚĆ WERYFIKACYJNA (SCHEMA + DATA) ---
print("\n" + "="*50)
print("   PODGLĄD UTWORZONYCH TABEL PARQUET")
print("="*50)

try:
    # 1. Pobieramy listę folderów w katalogu wyjściowym
    output_contents = fs.listStatus(Path(output_base_dir))
    
    found_any = False

    for item in output_contents:
        if item.isDirectory():
            found_any = True
            dir_name = item.getPath().getName()
            full_path = item.getPath().toString()
            
            print(f"\n📂 TABELA: {dir_name}")
            print(f"   Ścieżka: {full_path}")
            
            try:
                # 2. Wczytujemy Parquet
                df_check = spark.read.parquet(full_path)
                
                # 3. POKAZUJEMY SCHEMAT (Typy kolumn)
                print("--- SCHEMAT DANYCH ---")
                df_check.printSchema()
                
                # 4. POKAZUJEMY DANE (Pierwsze 5 wierszy)
                print("--- PRZYKŁADOWE REKORDY (Top 5) ---")
                df_check.show(5, truncate=False)
                
            except Exception as e:
                print(f"   ❌ Błąd odczytu tej tabeli: {e}")
            
            print("-" * 40) # Linia oddzielająca tabele

    if not found_any:
        print(f"Nie znaleziono żadnych folderów w {output_base_dir}")

except Exception as e:
    print(f"Błąd dostępu do folderu wyjściowego: {e}")


   PODGLĄD UTWORZONYCH TABEL PARQUET

📂 TABELA: company_earnings
   Ścieżka: hdfs://node1/user/vagrant/static_data_parquet/company_earnings
--- SCHEMAT DANYCH ---
root
 |-- symbol: string (nullable = true)
 |-- report_date: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- ebitda: double (nullable = true)
 |-- net_income: double (nullable = true)
 |-- revenue: double (nullable = true)

--- PRZYKŁADOWE REKORDY (Top 5) ---
+------+-----------+----+----------+----------+----------+
|symbol|report_date|year|ebitda    |net_income|revenue   |
+------+-----------+----+----------+----------+----------+
|AAPL  |2025-09-30 |2025|1.44748E11|1.1201E11 |4.16161E11|
|AAPL  |2024-09-30 |2024|1.34661E11|9.3736E10 |3.91035E11|
|AAPL  |2023-09-30 |2023|1.2582E11 |9.6995E10 |3.83285E11|
|AAPL  |2022-09-30 |2022|1.30541E11|9.9803E10 |3.94328E11|
|AAPL  |2021-09-30 |2021|null      |null      |null      |
+------+-----------+----+----------+----------+----------+
only showing top 5 rows

-

In [9]:
# finnhub_parquet

df_fp = spark.read.parquet("/user/vagrant/finnhub_parquet")
df_fp.show(5)

+---------------+--------+-------+-------------+----------+-------+--------------+
|         symbol|   price| volume|  trade_ts_ms|event_type| source|received_at_ms|
+---------------+--------+-------+-------------+----------+-------+--------------+
|BINANCE:BTCUSDT| 91185.7| 6.0E-5|1767974045464|     trade|finnhub| 1767974046026|
|BINANCE:BTCUSDT|91185.69|0.00219|1767974045464|     trade|finnhub| 1767974046027|
|BINANCE:ETHUSDT| 3111.45| 0.0017|1767974045467|     trade|finnhub| 1767974046027|
|BINANCE:BTCUSDT|91185.69|0.00127|1767974045469|     trade|finnhub| 1767974046027|
|BINANCE:BTCUSDT|91185.69| 2.2E-4|1767974045469|     trade|finnhub| 1767974046027|
+---------------+--------+-------+-------------+----------+-------+--------------+
only showing top 5 rows

